## __Phase 4b: Baseline comparison (PHMM Forward vs. Nearest-Neighbor)__

Compares the PHMM Forward alignment against a naive nearest-neighbor baseline, run with
four distance metrics as an ablation: plain Euclidean, Haversine, point-to-segment, and
time-weighted (see `scripts/modeling/AIS_RF_alignment/AIS_RF_nn_baseline.py`), plus a
random-choice floor (uniformly picking one candidate per RF signal) to show whether
matching beats chance at all.

PHMM Forward is scored with its length-corrected score (see
`scripts/evaluation/phmm_length_correction.py`): alpha is tuned on a 20% held-out split
of multimatch RF signals, applied to the other 80%, and every model here -- not just
PHMM Forward -- is restricted to that same 80% population (plus all single-match RF
signals) for a fair, apples-to-apples comparison.

The full comparison (metrics on both views, random baseline, runtime comparison,
ranking metrics, McNemar's test, bootstrap CIs) now lives in
`scripts/evaluation/run_phase4b_baseline_comparison.py`:

```bash
python -m scripts.evaluation.run_phase4b_baseline_comparison --error-model uniform
```

This notebook keeps a light interactive version for exploration only: it imports that
script's own compute functions (nothing duplicated here) and plots a couple of the
headline figures live via `scripts.evaluation.plots`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from scripts.evaluation import plots
from scripts.evaluation.baseline_comparison import (
    MODELS,
    build_corrected_phmm_results,
    evaluate_models,
    evaluate_ranking_models,
    load_model_results,
    random_choice_metrics,
    restrict_to_eval_population,
)

MODEL_NAMES = [name for name, *_ in MODELS]
RANDOM_BASELINE_NAME = "Random baseline"

In [ ]:
# Select which RF bearing-error model's data to compare (must match the
# --error-model used to generate/run the upstream scripts)
ERROR_MODEL = "uniform"  # "uniform" or "gaussian"
DATA_DIR = "../data/processed" if ERROR_MODEL == "uniform" else f"../data/processed/{ERROR_MODEL}"
BASE_DIR = "../data/processed"

df_preselection = pd.read_pickle(f"{DATA_DIR}/AIS_RF_preselection_data.pkl")
df_AIS_stats = pd.read_pickle(f"{BASE_DIR}/statistics_sample_5000.pkl").reset_index(drop=True)
results = load_model_results(DATA_DIR)

df_preselection_multimatch = df_preselection.groupby(
    ["RF_track_id", "RF_signal_id"]).filter(
        lambda x: x["AIS_track_id"].count() > 1).reset_index(drop=True)

corrected_phmm, best_alpha, df_preselection_multimatch_eval = build_corrected_phmm_results(
    df_preselection_multimatch, results["PHMM Forward"], df_AIS_stats
)
print(f"PHMM length-correction alpha (tuned on a 20% held-out split): {best_alpha}")
results["PHMM Forward"] = corrected_phmm

df_preselection_eval = restrict_to_eval_population(df_preselection, df_preselection_multimatch_eval)

In [ ]:
comparison_all = evaluate_models(df_preselection_eval, results)

random_all = random_choice_metrics(df_preselection_eval)
print(f"Random-choice baseline: precision={random_all['precision']:.4f} (std={random_all['precision_std']:.4f})")
comparison_all.loc[RANDOM_BASELINE_NAME] = {k: random_all[k] for k in comparison_all.columns}
comparison_all

### __Metrics chart (all candidate pairs, incl. random-choice floor)__

In [ ]:
plots.metric_comparison_bars(
    {"All candidate pairs": comparison_all}, MODEL_NAMES + [RANDOM_BASELINE_NAME],
    suptitle="PHMM Forward (corrected) vs. Nearest-Neighbor baselines vs. random-choice floor",
)
plt.show()

### __Score margin: PHMM Forward, correct vs incorrect top-1 picks__

In [ ]:
_, ranking_summaries_multi = evaluate_ranking_models(df_preselection_multimatch_eval, results)

summary = ranking_summaries_multi["PHMM Forward"].dropna(subset=["margin"]).copy()
top1_correct = summary["rank_of_true"] == 1

plots.score_margin_histogram(
    summary.loc[top1_correct, "margin"], summary.loc[~top1_correct, "margin"],
    title="PHMM Forward (corrected): score margin, correct vs incorrect top-1",
)
plt.show()